<a href="https://colab.research.google.com/github/harald-gen01/My_AI_learning_path/blob/main/Vibe_Coding_vs_Spec_Driven_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vibe Coding vs Spec-Driven Development
### A simple demo with the OpenAI API

## Executive takeaway

If you are using AI for software delivery, the strategic question is not only:

**How fast can we generate code?**

It is also:

**How clearly can we define what must be built before AI accelerates it?**

## TL;DR

This Colab compares the same coding task under two conditions:

1. **Vibe coding** → a vague prompt  
2. **Spec-driven development** → a clear specification

Both use the same model.

The goal is not to see which one writes more code.  
The goal is to see how **prompt clarity affects reliability, requirement coverage, and testability**.

### What this demonstrates
- Vibe coding is useful for exploration
- Spec-driven development is better for predictable execution
- The key difference is not model quality
- The key difference is **how much ambiguity exists before coding begins**

## Key takeaway

This is the main lesson:

- **Vibe coding** is often useful for exploration
- **Spec-driven development** is often better for reliability
- The difference is not just about code generation speed
- The difference is about **how much ambiguity exists before the model starts coding**

In other words:

**Better specifications create better execution conditions for AI.**



In [1]:
!pip -q install openai

In [6]:
import os
from openai import OpenAI

In [8]:
from google.colab import userdata

# Retrieve and set the OpenAI API key
OPENAI_API_KEY = userdata.get('openai_api_key')

client = OpenAI()
MODEL = "gpt-5"

## The task

We will ask the model to write the same Python function in two different ways.

**Target function:** `validate_login(payload)`

We want to compare:
- requirement coverage
- consistency
- error handling
- testability

In [10]:
SYSTEM_INSTRUCTIONS = """
You are a senior Python engineer.
Return only valid Python code.
Do not include markdown fences.
"""

vibe_prompt = """
Build a Python function called validate_login(payload) for a web app.
It should validate login input and handle errors.
"""

spec_prompt = """
Write a Python function named validate_login(payload).

Requirements:
- Input must be a dictionary.
- Required fields: username, password.
- If payload is not a dict, return:
  {"status": "error", "code": "invalid_payload"}
- If username is missing or empty, return:
  {"status": "error", "code": "missing_username"}
- If password is missing or empty, return:
  {"status": "error", "code": "missing_password"}
- If both username and password are present and non-empty, return:
  {"status": "success", "code": "login_ok"}
- Do not connect to a database.
- Do not use external libraries.
- Return only Python code.
"""

## Helper function

This helper sends a prompt to the model and returns the generated Python code.

In [11]:
def generate_code(instructions: str, prompt: str) -> str:
    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=prompt,
    )
    return response.output_text

## Generate both versions

First, a vague prompt ("vibe coding").  
Then, a clear specification ("spec-driven development").

In [12]:
vibe_code = generate_code(SYSTEM_INSTRUCTIONS, vibe_prompt)
spec_code = generate_code(SYSTEM_INSTRUCTIONS, spec_prompt)

print("=== VIBE CODING OUTPUT ===\n")
print(vibe_code)

print("\n" + "=" * 80 + "\n")

print("=== SPEC-DRIVEN OUTPUT ===\n")
print(spec_code)

=== VIBE CODING OUTPUT ===

import re
from collections.abc import Mapping

_EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
_USERNAME_RE = re.compile(r"^[A-Za-z0-9_.-]+$")

MAX_USERNAME_LEN = 150
MAX_EMAIL_LEN = 254
MAX_PASSWORD_LEN = 4096
MIN_IDENTIFIER_LEN = 1  # At least 1 visible char after strip
MAX_OTP_LEN = 8  # common 6-8 digits


def _add_error(errors, field, code, message):
    errors.setdefault(field, []).append({"code": code, "message": message})


def _coerce_str(value):
    if value is None:
        return None
    if isinstance(value, (bytes, bytearray)):
        try:
            value = value.decode("utf-8", errors="strict")
        except Exception:
            return None
    if not isinstance(value, str):
        # Try a simple cast for numbers/bools
        try:
            value = str(value)
        except Exception:
            return None
    # Normalize whitespace at ends
    s = value.strip()
    # Disallow embedded NULs
    if "\x00" in s:
        return 

## Load the generated code

We execute each model output in a separate namespace so we can test them independently.

In [13]:
vibe_ns = {}
spec_ns = {}

try:
    exec(vibe_code, vibe_ns)
except Exception as e:
    print("Error executing vibe_code:", e)

try:
    exec(spec_code, spec_ns)
except Exception as e:
    print("Error executing spec_code:", e)

vibe_fn = vibe_ns.get("validate_login")
spec_fn = spec_ns.get("validate_login")

print("Vibe function found:", callable(vibe_fn))
print("Spec function found:", callable(spec_fn))

Vibe function found: True
Spec function found: True


## Define shared test cases

Both versions will be evaluated against the exact same expected behavior.

In [14]:
test_cases = [
    (
        "missing username",
        {"password": "secret"},
        {"status": "error", "code": "missing_username"}
    ),
    (
        "missing password",
        {"username": "harald"},
        {"status": "error", "code": "missing_password"}
    ),
    (
        "valid input",
        {"username": "harald", "password": "secret"},
        {"status": "success", "code": "login_ok"}
    ),
    (
        "invalid payload",
        "not a dict",
        {"status": "error", "code": "invalid_payload"}
    ),
]

## Test runner

This function checks whether each implementation meets the expected contract.

In [15]:
def run_tests(fn, name):
    print(f"\n=== Testing: {name} ===")

    if not callable(fn):
        print("Function not found.")
        return 0

    passed = 0

    for label, payload, expected in test_cases:
        try:
            result = fn(payload)
            ok = result == expected
        except Exception as e:
            result = f"EXCEPTION: {type(e).__name__}: {e}"
            ok = False

        print(f"\nTest: {label}")
        print("Input:   ", payload)
        print("Expected:", expected)
        print("Actual:  ", result)
        print("PASS" if ok else "FAIL")

        if ok:
            passed += 1

    print(f"\n{name} score: {passed}/{len(test_cases)}")
    return passed

## Run the comparison

In [16]:
vibe_score = run_tests(vibe_fn, "Vibe Coding")
spec_score = run_tests(spec_fn, "Spec-Driven Development")


=== Testing: Vibe Coding ===

Test: missing username
Input:    {'password': 'secret'}
Expected: {'status': 'error', 'code': 'missing_username'}
Actual:   (False, {'errors': {'identifier': [{'code': 'required', 'message': 'An email or username is required.'}]}})
FAIL

Test: missing password
Input:    {'username': 'harald'}
Expected: {'status': 'error', 'code': 'missing_password'}
Actual:   (False, {'errors': {'password': [{'code': 'required', 'message': 'Password is required.'}]}})
FAIL

Test: valid input
Input:    {'username': 'harald', 'password': 'secret'}
Expected: {'status': 'success', 'code': 'login_ok'}
Actual:   (True, {'login_type': 'username', 'identifier': 'harald', 'password': 'secret', 'remember': False, 'otp': None})
FAIL

Test: invalid payload
Input:    not a dict
Expected: {'status': 'error', 'code': 'invalid_payload'}
Actual:   (False, {'errors': {'non_field_errors': [{'code': 'invalid_payload', 'message': 'Invalid input format.'}]}})
FAIL

Vibe Coding score: 0/4

=== 

## Simple summary

In [17]:
print("=== FINAL COMPARISON ===")
print(f"Vibe Coding score: {vibe_score}/{len(test_cases)}")
print(f"Spec-Driven score: {spec_score}/{len(test_cases)}")

if spec_score > vibe_score:
    print("\nSpec-driven development performed better on requirement coverage and testability.")
elif vibe_score > spec_score:
    print("\nVibe coding performed better in this run.")
else:
    print("\nBoth approaches performed equally in this run.")

=== FINAL COMPARISON ===
Vibe Coding score: 0/4
Spec-Driven score: 4/4

Spec-driven development performed better on requirement coverage and testability.
